# 01 · API와 메시지
**블록 2 · API와 메시지** — 실습 25분

1. 첫 호출 — SDK를 그대로 써 본다
2. 응답 읽기 — `stop_reason` · `usage`
3. system과 user
4. 모델은 기억하지 않는다 — 대화 이력
5. 같은 질문 다섯 번 — 그리고 temperature
6. 토큰과 비용

In [ ]:
import os
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir("..")          # day 폴더로 이동 — data/ · prompts/ · src/ 가 여기 있다

from common import llm      # 공용 API 도구 (ETRI_LLM_3days/common/llm.py)
print("모델:", llm.MODEL, "| 모의 모드" if llm.MOCK else "| 실제 호출")

## 1. 첫 호출
`src/llm.py`의 `call()`은 아래 코드를 감싼 것입니다. 한 번은 SDK를 그대로 써 봅니다.
(모의 모드에서는 이 셀을 건너뜁니다.)

In [ ]:
if not llm.MOCK:
    import anthropic
    client = anthropic.Anthropic()          # .env의 ANTHROPIC_API_KEY를 읽는다
    r = client.messages.create(
        model=llm.MODEL,
        max_tokens=300,
        messages=[{"role": "user", "content": "연차 신청 절차를 세 단계로 알려줘"}],
    )
    print(r.content[0].text)
    print("\n--- r에 저장된 것 전체 (글은 그중 하나일 뿐) ---")
    print(r)
else:
    print("모의 모드 — 건너뜀")

`r`은 글 하나가 아니라 `id`·`role`·`type`·`content`(블록 리스트)·`stop_reason`·`stop_sequence`·`model`·`usage`(입력/출력 토큰 · 캐시 사용량 · 처리 등급 등)·`container`·`diagnostics`·`stop_details`까지 다 들어있는 객체입니다. `r.content[0].text`는 그중 **글자만** 꺼낸 것이고, 나머지는 뒤에서(2·4절) 하나씩 다룹니다.

## 2. 응답 읽기
응답에는 글 말고도 **왜 멈췄는지**(`stop_reason`)와 **토큰을 얼마나 썼는지**(`usage`)가 들어 있습니다.

In [ ]:
r = llm.call("연차 신청 절차를 세 단계로 알려줘", max_tokens=300)
print(r.text)
print("---")
print("stop_reason :", r.stop_reason)
print("입력 토큰   :", r.input_tokens)
print("출력 토큰   :", r.output_tokens)
print("걸린 시간   :", f"{r.seconds:.2f}초")

### 입력은 두 번 모양이 바뀐다 — 내 코드 → 서버 → 모델
"안녕"이 모델에 닿기까지 **두 단계**를 거치고, 단계마다 모양이 다릅니다. 이 둘을 섞으면 토큰 수가 이해가 안 됩니다.

```
내 코드 ──(JSON)──▶ Anthropic 서버 ──(토큰 열)──▶ 모델
```

**1단계 — 내 코드가 서버로 보내는 것 (JSON).** 이건 모델이 받는 모양이 **아닙니다.**

In [ ]:
import json as _json

request = {
    "model": llm.MODEL,
    "max_tokens": 20,
    "messages": [{"role": "user", "content": "안녕"}],
}
print("내가 입력한 건: '안녕' (2글자)")
print("\n내 코드가 서버로 보내는 요청 (JSON) — 모델이 받는 모양은 아님:")
print(_json.dumps(request, ensure_ascii=False, indent=2))

**2단계 — 서버가 이 JSON을 읽고 모델용으로 다시 쓴 것.** 서버는 JSON 글자를 모델에 넘기지 않습니다.
- `model` → 어느 모델에 보낼지 **고르는 데만** 쓰고 버림. 모델 입력에 안 들어감
- `max_tokens` → 생성을 몇 토큰에서 **끊을지**에만 쓰고 버림. 모델 입력에 안 들어감
- `messages` → **이것만** 모델에 들어감. 단, `"role": "user"` 같은 글자가 아니라 **특수 토큰**(구조 표식)으로 바뀜

모델이 실제로 받는 토큰 열은 이런 모양입니다 (`[ ]`는 글자가 아닌 특수 토큰):
```
[대화 시작] [user 차례] 안녕 [내용 끝] [assistant 차례] ← 모델은 여기서부터 이어 쓴다
```

In [ ]:
r = llm.call("안녕", max_tokens=20)
print("모델이 받은 입력 토큰 수:", r.input_tokens, " — '안녕' 2글자가 11토큰")
print("\n돌아온 응답 전체:")
print(r.raw)

응답엔 `id`(요청 번호) · `stop_reason`(멈춘 이유) · `usage`(토큰 수뿐 아니라 캐시 사용량 · 처리 등급 · 추론 지역까지) 같은 게 글 말고도 같이 옵니다. `r.text`만 보면 이 중 대부분을 놓치는 겁니다 — `llm.py`의 `Result`는 지금 당장 필요한 몇 개(`text`·`stop_reason`·토큰 수)만 추려서 보여주는 것뿐입니다.

### 그 11토큰은 뭔가 — JSON 글자가 아니다
"JSON에서 안녕을 뺀 나머지가 토큰 아니냐"고 생각하기 쉽습니다. 아닙니다. 내용을 바꿔 가며 세어 보면 알 수 있습니다.

In [ ]:
for t in ["a", "hello", "안", "안녕"]:
    print(f"{t!r:34} → {llm.count_tokens(t):>2} 토큰")
j = '{"role": "user", "content": ""}'
print(f"{j!r:34} → {llm.count_tokens(j):>2} 토큰   ← JSON 글자를 내용으로 보내 보면")

- 영어는 `a`든 `hello`든 **단어 하나 = 1토큰**이라 전부 8 → 내용을 뺀 **틀이 정확히 7개**
- 한글은 **한 글자 ≈ 2토큰** → `안` 7+2=9, `안녕` 7+4=11
- JSON 글자 `{"role": "user", "content": ""}`를 **내용으로** 보내면 틀 7을 빼고도 10개가 넘습니다. JSON 글자가 틀이었다면 7이 나올 수 없습니다 → **틀 7개는 JSON 글자가 아닙니다**

그러니 11개 = **구조 표식 7개 + "안녕" 4개**입니다.

### 틀 7개는 "내 메시지 포장 3 + 대화 틀 4"
`role`·`user`·`content`에 해당하는 표식이라면 3개면 될 텐데 왜 7인가 — 메시지 수를 바꿔 가며 재면 갈립니다.

In [ ]:
import anthropic
_c = anthropic.Anthropic()
def _n(msgs):
    return _c.messages.count_tokens(model=llm.MODEL, messages=msgs).input_tokens
u = lambda s: {"role": "user", "content": s}
a = lambda s: {"role": "assistant", "content": s}
for name, msgs in [("1턴 [a]", [u("a")]),
                   ("3턴 [a|b|a]", [u("a"), a("b"), u("a")]),
                   ("5턴 [a|b|a|b|a]", [u("a"), a("b"), u("a"), a("b"), u("a")])]:
    total = _n(msgs); content = len(msgs)      # 내용은 메시지당 1토큰
    print(f"{name:18} → {total:>2} 토큰 = 내용 {content} + 틀 {total - content}")

메시지가 하나 늘 때마다 틀이 **정확히 3**씩 늡니다 → `틀 = 4 + 3 × 메시지 수`.

| 개수 | 정체 | 언제 붙나 |
|---|---|---|
| **3** | 내 메시지 하나의 포장 — 턴 시작 · 누구 차례(user/assistant) · 턴 끝. `"role": "user", "content":`가 뜻하는 바로 그것 | 메시지 1개마다 |
| **4** | 대화 틀 — 토큰 열 **시작** 표시 + 맨 끝의 **"이제 assistant 차례"** 표시. 후자는 내 JSON에 없습니다 — 응답이 assistant 턴이라 서버가 "여기서부터 네가 써"라고 열어 주는 것 | 대화 전체에 1번 |

```
[대화 시작] [user 차례] 안녕 [내용 끝] [assistant 차례] ← 모델이 이어 쓴다
```

내가 쓴 글자(2개)보다 이 표식(7개)이 더 크고, 한글은 영어보다 글자당 비쌉니다 — 글자 수로 토큰 수를 어림잡으면 안 되는 이유입니다. 그리고 4절에서 보겠지만, 턴이 쌓이면 이 "메시지당 3개"가 매번 전부 다시 들어갑니다.

### max_tokens에서 잘리면
`max_tokens`를 작게 주면 글이 중간에 끊기고 `stop_reason`이 `max_tokens`가 됩니다.
**프로그램은 글을 쓰기 전에 `stop_reason`부터 확인합니다** — 잘린 JSON은 파싱이 깨집니다.

In [ ]:
r = llm.call("연차 신청 절차를 세 단계로 자세히 알려줘", max_tokens=20)
print(repr(r.text))
print("stop_reason :", r.stop_reason)
if r.stop_reason == "max_tokens":
    print("→ 잘렸습니다. max_tokens를 늘리거나 더 짧게 쓰라고 지시합니다.")

## 3. system과 user
매번 같은 **역할 · 규칙 · 형식**은 `system`에, 매번 바뀌는 **자료와 질문**은 `user`에 둡니다.
같은 질문에 system만 바꿔 봅니다.

In [ ]:
question = "다음 주 화요일 오후 2시에 전 직원 소방 훈련이 있습니다. 이 내용을 안내해 주세요."

for system in [
    None,
    "당신은 신입 사원에게 친절하게 설명하는 인사팀 담당자입니다. 두 문장 이내로 씁니다.",
    "당신은 사내 메신저 공지 봇입니다. 이모지 없이 한 줄로, '[공지]'로 시작합니다.",
]:
    r = llm.call(question, system=system, max_tokens=200)
    print(f"[system] {system}\n{r.text.strip()}\n")

## 4. 모델은 기억하지 않는다
API는 매 호출이 처음입니다. 이어서 대화하려면 **앞서 주고받은 메시지를 직접 다시 보냅니다** — 그래서 턴이 늘어날수록 **매번 보내는 요청 자체가 커집니다.**

아래 셀은 2턴짜리 대화를 돌리면서, **매 턴마다 서버로 나가는 요청(JSON) 전체와 돌아오는 응답 전체**를 빠짐없이 찍습니다. (2절에서 봤듯 모델은 이 JSON의 `messages`만 특수 토큰으로 바꿔 받습니다.)

In [ ]:
import json

history = []
sent = []          # 턴마다 (보낸 messages, 응답) 을 남겨 둔다 — 아래에서 토큰 수를 검산한다
turns = [
    "제 이름은 김민수이고, 생산관리팀에서 일합니다.",
    "제 이름과 부서가 뭐였죠?",
]
for i, user_text in enumerate(turns, 1):
    messages = history + [{"role": "user", "content": user_text}]
    request = {"model": llm.MODEL, "max_tokens": 100, "messages": messages}
    print(f"===== {i}번째 턴 =====")
    print("--- 서버로 나가는 요청 (JSON) ---")
    print(json.dumps(request, ensure_ascii=False, indent=2))

    r = llm.call(user_text, history=history, max_tokens=100)
    sent.append((messages, r))

    print("\n--- 돌아온 응답 전체 ---")
    print(r.raw)
    print()

    history = messages + [{"role": "assistant", "content": r.text}]

**1턴**은 `messages`에 내가 보낸 말 하나뿐이지만, **2턴**은 1턴의 내 말 + 모델의 답 + 이번 내 말, **세 개**가 들어갑니다. 이력 없이 "제 이름과 부서가 뭐였죠?"만 물으면 모델은 모릅니다 — 매번 **지금까지 오간 것 전부**를 다시 보내야 기억하는 것처럼 보입니다.

### 멀티턴 입력 토큰은 이 식으로 정확히 맞는다
2절에서 메시지 하나의 포장이 **3**, 대화 틀이 **4**였습니다. 턴이 쌓이면:

```
input_tokens = 4 + Σ (3 + 메시지 내용 토큰)     ← 지금까지 쌓인 모든 메시지에 대해
```

아래 셀이 방금 돌린 두 턴의 실제 `usage.input_tokens`와 이 식을 대조합니다. (메시지 하나의 내용 토큰 = 그것만 따로 센 값 − 7)

In [ ]:
content = lambda s: llm.count_tokens(s) - 7      # 메시지 1개 측정값 − 틀 7 = 순수 내용 토큰

for i, (messages, r) in enumerate(sent, 1):
    pieces = [content(m["content"]) for m in messages]
    pred = 4 + sum(3 + p for p in pieces)
    ok = "✓" if pred == r.input_tokens else "✗"
    print(f"{i}턴: 메시지 {len(messages)}개, 내용 토큰 {pieces}")
    print(f"     4 + Σ(3+내용) = {pred}   실제 input_tokens = {r.input_tokens}   {ok}")

2턴째 내용 토큰 가운데 가장 큰 덩어리는 **1턴에서 모델이 출력한 답**입니다. 1턴엔 `output_tokens`로 냈던 것을 2턴엔 `input_tokens`로 **다시** 냅니다 — 멀티턴에서 비용이 빨리 불어나는 진짜 이유입니다. 메시지당 틀 3개도 매번 전부 다시 들어갑니다.

**요청보다 응답에 훨씬 많은 게 따라옵니다.** 요청은 `model`·`max_tokens`·`messages`, 딱 3가지뿐이었습니다. 방금 받은 마지막 응답(`r`) 하나를 열어서, 거기 뭐가 더 들어있는지 하나씩 짚어 봅니다.

In [ ]:
u = r.raw.usage
block = r.raw.content[0]      # 우리가 r.text로 쓰는 게 바로 이 블록의 .text
print("요청에는 모델 이름 · max_tokens · messages, 이 3가지만 넣었습니다.")
print("마지막 턴의 응답(r.raw)에는 그보다 훨씬 많은 게 같이 옵니다:\n")
print(f"  content                           : (길이 {len(r.raw.content)}인 리스트)")
print(f"      답이 블록 하나가 아니라 **리스트**인 이유 — 도구를 쓰면 블록이 여러 개(생각 ·")
print(f"      도구 호출 · 텍스트)로 늘어납니다. 지금은 글만 와서 블록이 하나뿐입니다.")
print(f"  content[0].type                  : {block.type}")
print(f"      이 블록이 글(text)인지 · 생각(thinking)인지 · 도구 호출(tool_use)인지")
print(f"  content[0].text                  : {block.text[:40]!r}...")
print(f"      우리가 r.text로 꺼내 쓰는 바로 이 값 — content[0].text")
print(f"  content[0].citations              : {block.citations}")
print(f"      문서를 인용하며 답했을 때만 출처가 들어감 (지금은 없음)")
print(f"  role                              : {r.raw.role}")
print(f"      누가 말했는지 — 응답은 항상 'assistant'")
print(f"  type                              : {r.raw.type}")
print(f"      이 객체 자체가 무엇인지 — 'message'")
print(f"  container                         : {r.raw.container}")
print(f"      코드 실행 도구를 쓸 때만 생기는 실행 환경 정보 (안 쓰면 None)")
print(f"  diagnostics                       : {r.raw.diagnostics}")
print(f"      캐시 진단(베타 기능)을 켰을 때만 옴 (안 켜면 None)")
print(f"  stop_details                      : {r.raw.stop_details}")
print(f"      거절(refusal)당했을 때만 이유가 들어감 (지금처럼 정상 답이면 None)")
print(f"  id                                : {r.raw.id}")
print(f"      내가 보낸 적 없음 — 이 응답 하나를 가리키는 고유 번호")
print(f"  stop_reason                       : {r.raw.stop_reason}")
print(f"      내가 물은 적 없음 — 왜 멈췄는지(end_turn · max_tokens · tool_use · ...)")
print(f"  stop_sequence                     : {r.raw.stop_sequence}")
print(f"      어떤 중단 문자열에 걸렸는지 (안 걸리면 None)")
print(f"  model                             : {r.raw.model}")
print(f"      실제로 처리한 모델 — 요청에 쓴 이름과 같아야 정상")
print(f"  usage.input_tokens / output_tokens: {u.input_tokens} / {u.output_tokens}")
print(f"      우리가 Result로 뽑아 쓰는 바로 그 숫자")
print(f"  usage.cache_creation_input_tokens : {u.cache_creation_input_tokens}")
print(f"      프롬프트 캐시에 이번에 새로 쓴 토큰 수 (지금은 캐시를 안 써서 0)")
print(f"  usage.cache_read_input_tokens     : {u.cache_read_input_tokens}")
print(f"      캐시에서 그냥 읽어서 싸게 처리한 토큰 수")
print(f"  usage.service_tier                : {u.service_tier}")
print(f"      이번 요청이 처리된 등급")
print(f"  usage.inference_geo               : {u.inference_geo}")
print(f"      추론이 실제로 일어난 지역")

요청 쪽은 3가지뿐인데, 응답 쪽은 이렇게 **15가지가 넘는 부가 정보**가 따라옵니다. `content`조차 글 하나가 아니라 **블록 리스트**이고, 우리가 늘 쓰는 `r.text`는 사실 그 리스트의 첫 블록(`content[0]`)의 `.text`일 뿐입니다. `r.text`만 보면 이 중 거의 전부를 놓치는 겁니다 — `llm.py`의 `Result`는 당장 쓸모 있는 몇 개(`text`·`stop_reason`·토큰 수)만 추려 둔 것뿐이고, 나머지는 `r.raw`를 직접 열어야 보입니다.

## 5. 같은 질문 다섯 번
같은 질문을 두 가지로 각각 5회 보내고 결과를 나란히 놓습니다.

**관찰할 것**
- 다섯 개 중 몇 개가 서로 같은가
- 달라지는 곳은 문장의 어디인가
- 정답이 거의 정해진 질문(절차)과 열린 질문(슬로건)은 차이가 어떻게 다른가
- 출력 토큰 수는 매번 같은가

In [ ]:
QUESTIONS = {
    "절차": "회사 연차 신청 절차를 세 단계로 알려줘",
    "슬로건": "신제품 무선 청소기 슬로건 하나만 지어줘",
}

runs = {}
for name, q in QUESTIONS.items():
    runs[name] = []
    for i in range(5):
        r = llm.call(q, max_tokens=200)
        runs[name].append({"text": r.text, "output_tokens": r.output_tokens})
        print(name, i + 1, r.output_tokens, "|", r.text.strip().replace("\n", " ")[:60])
    print()

for name, rs in runs.items():
    print(f"{name}: 서로 다른 답 {len({x['text'] for x in rs})}개 / 5")

### temperature — 흔들림을 조절하는 손잡이
`temperature`는 다음 토큰을 얼마나 고르게 뽑을지 정합니다. **0 = 가장 확률 높은 토큰 위주(덜 흔들림)**, **1 = 더 고르게(더 다양함)**.
- **SDK 1.x**: `messages.create()`에서 `temperature` 인자가 빠졌습니다. (`llm.call`은 `extra_body`로 우회해 보냅니다.)
- **API**: 모델마다 다릅니다. 이전 세대(Haiku 4.5)는 받지만, **최신 모델(Sonnet 5.5 · Opus 5.5)은 거부합니다** — 400 오류.

아래 셀은 ① Haiku로 같은 질문을 온도 0과 1에서 세 번씩 보내 비교하고, ② Sonnet 5.5에 보내 거부되는 것을 봅니다.

**관찰할 것**
- 온도 1에서 답이 얼마나 다양해지는가
- 온도 0이면 세 번 모두 똑같은가 — *0이어도 완전히 같다는 보장은 없습니다*
- 최신 모델에서는 이 손잡이 자체가 없다 → 그래서 **여러 번 돌려 재는 습관(블록 4)**이 필요하다

In [ ]:
Q = "신제품 무선 청소기 슬로건 하나만 지어줘. 슬로건만 답해."

# ① temperature를 받는 모델(Haiku 4.5) — 온도를 바꿔 가며 세 번씩
print("모델:", llm.MODEL)
for t in [0.0, 1.0]:
    answers = [llm.call(Q, temperature=t, max_tokens=60).text.strip() for _ in range(3)]
    print(f"\ntemperature={t} → 서로 다른 답 {len(set(answers))}개 / 3")
    for a in answers:
        print("   ", a.replace("\n", " ")[:60])

# ② 최신 모델(Sonnet 5.5)은 temperature 자체를 거부한다
print("\n모델:", llm.MODEL_ADVANCED)
try:
    llm.call(Q, model=llm.MODEL_ADVANCED, temperature=0.0, max_tokens=20)
    print("받아들였습니다")
except Exception as e:
    print(type(e).__name__, "—", str(e)[:200])

> 대신 생길 수 있는 조절 손잡이는 `effort`("low"~"max")처럼 *얼마나 생각할지*입니다. 지원 여부는 모델마다 다르므로 강의 당일 문서로 확인합니다.

결과는 지우지 말고 저장해 둡니다. 오후 블록 4에서 *한 번 돌려 보고 판단하면 안 되는 이유*의 근거로 다시 씁니다.

In [ ]:
import json
Path("results").mkdir(exist_ok=True)
Path("results/repeat5.json").write_text(
    json.dumps({"questions": QUESTIONS, "runs": runs}, ensure_ascii=False, indent=2),
    encoding="utf-8")
print("저장: results/repeat5.json")

## 6. 토큰과 비용
비용 = 입력 토큰 × 입력 단가 + 출력 토큰 × 출력 단가. **단가는 강의 당일 요금표로 `src/llm.py`의 `PRICES`를 채웁니다.**
(채우지 않은 모델은 "단가 미입력"으로 나옵니다.)

In [ ]:
doc = Path("data/docs/11_정보보호지침_개정_긴문서.txt").read_text(encoding="utf-8")
r = llm.call(f"다음 문서를 세 줄로 요약해 주세요.\n\n{doc}", max_tokens=300)
print(r.text.strip(), "\n")
print(f"입력 {r.input_tokens} 토큰 · 출력 {r.output_tokens} 토큰 · 1건 {llm.fmt_cost(r.cost)}")
if r.cost is not None:
    print(f"하루 1,000건이면 약 ${r.cost * 1000:.2f}")